# AlpCAN CT Pipeline: Nodül Segmentasyon Eğitimi

**Amaç:** LIDC-IDRI veri seti üzerinde U-Net tabanlı nodül segmentasyon modeli eğitmek.  
AlpCAN CT Pipeline Agent 2 (Nodül Tespit) için ağırlık üretimi.

**İçindekiler:**
1. LIDC-IDRI veri seti yükleme ve konsünsüs maskeleri
2. Eğitim/doğrulama bölmesi (hasta bazlı)
3. U-Net mimari tanımlama (ResNet-34 encoder)
4. Veri artırma (augmentation)
5. Eğitim (Dice + BCE loss, 50 epoch)
6. Doğrulama metrikleri (Dice, IoU, Precision, Recall)
7. Connected component analizi ile nodül çıkarımı
8. Sonuçları kaydetme + ağırlık export

**Model:** U-Net (ResNet-34 encoder, ImageNet pre-trained)  
**GPU:** Kaggle T4 16GB  
**Dataset:** `zhangweiled/lidcidri` -- LIDC-IDRI kesilmiş nodül dilimleri

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

print(f"PyTorch: {torch.__version__}")
print(f"SMP: {smp.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. LIDC-IDRI Veri Seti Yükleme

In [ ]:
# Dataset yolunu bul
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LIDC-IDRI dataset
DATA_DIR = Path("/kaggle/input/lidcidri/LIDC-IDRI-slices")
if not DATA_DIR.exists():
    # Alternatif yol ara
    found = False
    for candidate in INPUT_DIR.rglob("LIDC-IDRI-0001"):
        DATA_DIR = candidate.parent
        found = True
        break
    if not found:
        raise FileNotFoundError(
            "LIDC-IDRI dataset bulunamadi! "
            "'zhangweiled/lidcidri' dataset'ini Kaggle notebook'a ekleyin."
        )

print(f"Dataset: {DATA_DIR}")

# Tum hasta ve nodul bilgilerini topla
patient_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Toplam hasta dizini: {len(patient_dirs)}")

# Her nodul icin dilim ve maske bilgisi
all_samples = []
skipped = 0

for pdir in patient_dirs:
    nodule_dirs = sorted([d for d in pdir.iterdir() if d.is_dir()])
    for ndir in nodule_dirs:
        img_dir = ndir / "images"
        mask_dirs = sorted(
            [d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")]
        )

        if not img_dir.exists() or not mask_dirs:
            skipped += 1
            continue

        for img_file in sorted(img_dir.glob("*.png")):
            # Her annotator maskesini kontrol et
            masks_exist = []
            for mdir in mask_dirs:
                mask_file = mdir / img_file.name
                if mask_file.exists():
                    masks_exist.append(mask_file)

            if masks_exist:
                all_samples.append({
                    'patient_id': pdir.name,
                    'nodule': ndir.name,
                    'image_path': str(img_file),
                    'mask_paths': [str(m) for m in masks_exist],
                    'n_annotators': len(masks_exist),
                })

df = pd.DataFrame(all_samples)
print(f"\nToplam dilim (egitim ornegi): {len(df):,}")
print(f"Toplam hasta: {df['patient_id'].nunique()}")
print(f"Toplam nodul: {df.groupby(['patient_id', 'nodule']).ngroups}")
print(f"Atlanan nodul dizini: {skipped}")
print(f"\nAnnotator dagilimi:")
print(df['n_annotators'].value_counts().sort_index())

---
## 2. Eğitim/Doğrulama Bölmesi (Hasta Bazlı)

In [ ]:
# Hasta bazli 80/20 bolme (veri sizintisi onleme)
np.random.seed(42)
patients = df['patient_id'].unique().copy()
np.random.shuffle(patients)

split_idx = int(len(patients) * 0.8)
train_patients = set(patients[:split_idx])
val_patients = set(patients[split_idx:])

train_df = df[df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['patient_id'].isin(val_patients)].reset_index(drop=True)

print(f"Train: {len(train_df):,} dilim ({len(train_patients)} hasta)")
print(f"Val:   {len(val_df):,} dilim ({len(val_patients)} hasta)")
print(f"\nTrain/Val oran: {len(train_df)/(len(train_df)+len(val_df)):.1%} / {len(val_df)/(len(train_df)+len(val_df)):.1%}")

---
## 3. Dataset ve Augmentation

In [ ]:
IMG_SIZE = 256

# Augmentation
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5
    ),
    A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.2),
    A.RandomBrightnessContrast(
        brightness_limit=0.2, contrast_limit=0.2, p=0.3
    ),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    ToTensorV2(),
])


class LIDCSegDataset(Dataset):
    """LIDC-IDRI nodul segmentasyon dataset'i.

    Konsensus maskesi: Tum annotator maskelerinin ortalamasi >= 0.5 threshold.
    Goruntu 3 kanala kopyalanir (grayscale -> RGB, ImageNet pretrained encoder icin).
    """

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Goruntu oku (grayscale -> 3 kanal)
        img = np.array(Image.open(row['image_path']).convert('L'), dtype=np.uint8)
        img_rgb = np.stack([img, img, img], axis=-1)  # (H, W, 3)

        # Konsensus maskesi olustur
        mask_sum = np.zeros(img.shape[:2], dtype=np.float32)
        n_masks = 0
        for mask_path in row['mask_paths']:
            try:
                m = np.array(Image.open(mask_path).convert('L'), dtype=np.uint8)
                # Boyut uyumsuzlugu kontrolu
                if m.shape[:2] != img.shape[:2]:
                    m = np.array(
                        Image.open(mask_path).convert('L').resize(
                            (img.shape[1], img.shape[0]), Image.NEAREST
                        ),
                        dtype=np.uint8,
                    )
                mask_sum += (m > 0).astype(np.float32)
                n_masks += 1
            except Exception:
                continue

        # Konsensus: annotatorlerin >= %50'si isaretlemis
        consensus = (mask_sum / max(n_masks, 1) >= 0.5).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=img_rgb, mask=consensus)
            img_tensor = augmented['image']       # (3, H, W)
            mask_tensor = augmented['mask']        # (H, W)
            # mask_tensor float32'ye cevirip kanal boyutu ekle
            mask_tensor = mask_tensor.float().unsqueeze(0)  # (1, H, W)
        else:
            img_tensor = torch.from_numpy(
                img_rgb.transpose(2, 0, 1)
            ).float() / 255.0
            mask_tensor = torch.from_numpy(consensus).unsqueeze(0).float()

        return img_tensor, mask_tensor


# Dataset olustur
train_dataset = LIDCSegDataset(train_df, transform=train_transform)
val_dataset = LIDCSegDataset(val_df, transform=val_transform)

BATCH_SIZE = 16
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"Train dataset: {len(train_dataset)} ornek")
print(f"Val dataset:   {len(val_dataset)} ornek")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Train batch:   {len(train_loader)}")
print(f"Val batch:     {len(val_loader)}")

# Ornek kontrol
img_sample, mask_sample = train_dataset[0]
print(f"\nGoruntu shape: {img_sample.shape}, dtype: {img_sample.dtype}")
print(f"Maske shape:   {mask_sample.shape}, dtype: {mask_sample.dtype}")
print(f"Maske unique:  {mask_sample.unique().tolist()}")

In [ ]:
# Ornek batch gorsellestirme
n_show = min(4, len(val_dataset))
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
if n_show == 1:
    axes = axes.reshape(2, 1)
fig.suptitle(
    'Egitim Ornekleri -- Goruntu + Konsensus Maske', fontsize=14
)

# Guvenli indeks secimi
step = max(1, len(val_dataset) // n_show)
show_indices = [min(i * step, len(val_dataset) - 1) for i in range(n_show)]

for col, idx in enumerate(show_indices):
    img, mask = val_dataset[idx]

    # Denormalize
    img_np = img.numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = (img_np * std + mean).clip(0, 1)

    axes[0, col].imshow(img_np[:, :, 0], cmap='gray')
    axes[0, col].set_title(f'Dilim idx={idx}')
    axes[0, col].axis('off')

    axes[1, col].imshow(img_np[:, :, 0], cmap='gray')
    mask_np = mask.numpy()[0]
    if mask_np.max() > 0:
        axes[1, col].contour(
            mask_np, colors='lime', linewidths=1.5, levels=[0.5]
        )
    axes[1, col].set_title('+ Konsensus Maske')
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'ct_seg_training_samples.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print(f"Kaydedildi: {OUTPUT_DIR / 'ct_seg_training_samples.png'}")

---
## 4. Model Tanımlama -- U-Net (ResNet-34 Encoder)

In [ ]:
# U-Net modeli -- ResNet-34 encoder (ImageNet pre-trained)
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,       # Binary segmentasyon (nodul / arkaplan)
    activation=None,  # Raw logits -- loss icinde sigmoid uygulanacak
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model:     U-Net (ResNet-34 encoder)")
print(f"Toplam parametre:  {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"Egitilecek:        {n_trainable:,}")
print(f"Device:            {device}")

---
## 5. Loss Fonksiyonu ve Optimizer

In [ ]:
class DiceBCELoss(nn.Module):
    """Dice Loss + BCE Loss kombinasyonu.

    Dice: Bolgesel overlap metrik (class imbalance'a dayanikli)
    BCE:  Piksel bazli binary cross-entropy
    """

    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def dice_loss(self, pred, target):
        pred_sigmoid = torch.sigmoid(pred)
        # Flatten spatial dims: (B, 1, H, W) -> sum over (H, W)
        intersection = (pred_sigmoid * target).sum(dim=(2, 3))
        union = pred_sigmoid.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

    def forward(self, pred, target):
        dice = self.dice_loss(pred, target)
        bce = self.bce(pred, target)
        return self.dice_weight * dice + self.bce_weight * bce


criterion = DiceBCELoss(dice_weight=0.5, bce_weight=0.5)

optimizer = optim.AdamW(
    model.parameters(), lr=1e-4, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50, eta_min=1e-6
)

print("Loss:      DiceBCE (0.5 Dice + 0.5 BCE)")
print("Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print("Scheduler: CosineAnnealing (T_max=50, eta_min=1e-6)")

---
## 6. Eğitim Döngüsü

In [ ]:
def compute_metrics(pred, target, threshold=0.5):
    """Segmentasyon metrikleri hesapla.

    Args:
        pred: (B, 1, H, W) raw logits
        target: (B, 1, H, W) binary mask
        threshold: sigmoid ciktisi icin esik

    Returns:
        dict: dice, iou, precision, recall degerleri
    """
    pred_binary = (torch.sigmoid(pred) > threshold).float()

    intersection = (pred_binary * target).sum(dim=(2, 3))
    pred_sum = pred_binary.sum(dim=(2, 3))
    target_sum = target.sum(dim=(2, 3))
    union = pred_sum + target_sum

    smooth = 1e-6
    dice = (2.0 * intersection + smooth) / (union + smooth)
    iou = (intersection + smooth) / (union - intersection + smooth)
    precision = (intersection + smooth) / (pred_sum + smooth)
    recall = (intersection + smooth) / (target_sum + smooth)

    return {
        'dice': dice.mean().item(),
        'iou': iou.mean().item(),
        'precision': precision.mean().item(),
        'recall': recall.mean().item(),
    }


NUM_EPOCHS = 100
best_val_dice = 0.0
best_epoch = -1
history = {
    'epoch': [],
    'train_loss': [],
    'val_loss': [],
    'val_dice': [],
    'val_iou': [],
    'val_precision': [],
    'val_recall': [],
    'lr': [],
}

# AMP -- Kaggle uyumlu (torch.cuda.amp kullan, torch.amp degil)
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print(f"Egitim basliyor -- {NUM_EPOCHS} epoch, AMP={'ON' if use_amp else 'OFF'}")
print("=" * 70)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    # --- Train ---
    model.train()
    train_loss = 0.0
    n_train = 0

    for images, masks in train_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        n_train += images.size(0)

    train_loss /= max(n_train, 1)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_metrics_accum = defaultdict(float)
    n_val = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, masks)

            val_loss += loss.item() * images.size(0)
            batch_metrics = compute_metrics(outputs.float(), masks.float())
            for k, v in batch_metrics.items():
                val_metrics_accum[k] += v * images.size(0)
            n_val += images.size(0)

    val_loss /= max(n_val, 1)
    val_metrics = {k: v / max(n_val, 1) for k, v in val_metrics_accum.items()}

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # History kaydet
    history['epoch'].append(epoch + 1)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_metrics.get('dice', 0.0))
    history['val_iou'].append(val_metrics.get('iou', 0.0))
    history['val_precision'].append(val_metrics.get('precision', 0.0))
    history['val_recall'].append(val_metrics.get('recall', 0.0))
    history['lr'].append(current_lr)

    # Best model kaydet
    current_dice = val_metrics.get('dice', 0.0)
    if current_dice > best_val_dice:
        best_val_dice = current_dice
        best_epoch = epoch + 1
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_dice': best_val_dice,
                'val_iou': val_metrics.get('iou', 0.0),
                'val_precision': val_metrics.get('precision', 0.0),
                'val_recall': val_metrics.get('recall', 0.0),
                'img_size': IMG_SIZE,
                'encoder': 'resnet34',
            },
            OUTPUT_DIR / 'ct_seg_best_model.pth',
        )
        marker = ' << BEST'
    else:
        marker = ''

    # Her 5 epoch'ta ve ilk epoch'ta rapor
    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        epoch_time = time.time() - epoch_start
        print(
            f"Epoch {epoch + 1:>3d}/{NUM_EPOCHS} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"Dice: {current_dice:.4f} | "
            f"IoU: {val_metrics.get('iou', 0):.4f} | "
            f"LR: {current_lr:.6f} | "
            f"{epoch_time:.0f}s | "
            f"Total: {elapsed / 60:.1f}m{marker}"
        )

total_time = time.time() - start_time
print(f"\n{'=' * 70}")
print(f"Egitim tamamlandi! Sure: {total_time / 60:.1f} dakika")
print(f"En iyi Val Dice: {best_val_dice:.4f} (Epoch {best_epoch})")


---
## 7. Eğitim Grafikleri

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], label='Val', linewidth=2)
axes[0].set_title('Loss (DiceBCE)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice & IoU
axes[1].plot(
    epochs_range, history['val_dice'],
    label='Dice', linewidth=2, color='#e74c3c'
)
axes[1].plot(
    epochs_range, history['val_iou'],
    label='IoU', linewidth=2, color='#3498db'
)
axes[1].plot(
    epochs_range, history['val_precision'],
    label='Precision', linewidth=2, color='#2ecc71', linestyle='--'
)
axes[1].plot(
    epochs_range, history['val_recall'],
    label='Recall', linewidth=2, color='#9b59b6', linestyle='--'
)
axes[1].set_title('Validasyon Metrikleri', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

# Learning Rate
axes[2].plot(
    epochs_range, history['lr'],
    linewidth=2, color='#2ecc71'
)
axes[2].set_title('Learning Rate (Cosine)', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'ct_seg_training_curves.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print(f"Kaydedildi: {OUTPUT_DIR / 'ct_seg_training_curves.png'}")

---
## 8. Doğrulama Sonuçları Görselleştirme

In [ ]:
# En iyi modeli yukle
checkpoint = torch.load(
    OUTPUT_DIR / 'ct_seg_best_model.pth',
    map_location=device,
    weights_only=False,
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(
    f"En iyi model yuklendi "
    f"(Epoch {checkpoint['epoch'] + 1}, Dice={checkpoint['val_dice']:.4f})"
)

# 8 ornek tahmin gorsellestir (veya daha az, dataset kucukse)
n_cols = min(8, len(val_dataset))
fig, axes = plt.subplots(3, n_cols, figsize=(3 * n_cols, 9))
if n_cols == 1:
    axes = axes.reshape(3, 1)
fig.suptitle(
    f'Validasyon Tahminleri -- Best Dice: {checkpoint["val_dice"]:.4f}',
    fontsize=14,
)

indices = np.linspace(0, len(val_dataset) - 1, n_cols, dtype=int)

with torch.no_grad():
    for col, idx in enumerate(indices):
        img, gt_mask = val_dataset[idx]
        img_gpu = img.unsqueeze(0).to(device)

        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = model(img_gpu)
        pred_mask = (torch.sigmoid(pred) > 0.5).cpu().numpy()[0, 0]

        # Denormalize
        img_np = img.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = (img_np * std + mean).clip(0, 1)

        gt_np = gt_mask.numpy()[0]

        # Satir 0: Goruntu
        axes[0, col].imshow(img_np[:, :, 0], cmap='gray')
        axes[0, col].set_title(f'Dilim {idx}', fontsize=9)
        axes[0, col].axis('off')

        # Satir 1: Ground Truth overlay
        axes[1, col].imshow(img_np[:, :, 0], cmap='gray')
        if gt_np.max() > 0:
            axes[1, col].contour(
                gt_np, colors='lime', linewidths=1.5, levels=[0.5]
            )
        axes[1, col].set_title('GT', fontsize=9)
        axes[1, col].axis('off')

        # Satir 2: Prediction overlay
        axes[2, col].imshow(img_np[:, :, 0], cmap='gray')
        if pred_mask.max() > 0:
            axes[2, col].contour(
                pred_mask.astype(np.float32),
                colors='red', linewidths=1.5, levels=[0.5]
            )
        axes[2, col].set_title('Tahmin', fontsize=9)
        axes[2, col].axis('off')

axes[0, 0].set_ylabel('Goruntu', fontsize=11)
axes[1, 0].set_ylabel('GT Maske', fontsize=11)
axes[2, 0].set_ylabel('Tahmin', fontsize=11)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'ct_seg_predictions.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print(f"Kaydedildi: {OUTPUT_DIR / 'ct_seg_predictions.png'}")

---
## 9. Connected Component Analizi -- Nodül Çıkarımı

In [ ]:
from scipy import ndimage


def extract_nodules_from_mask(pred_mask, min_area=10):
    """Binary maskeden bireysel nodulleri cikar.

    Args:
        pred_mask: (H, W) binary maske (numpy array)
        min_area: Minimum piksel alani (kucuk gurultu filtreleme)

    Returns:
        nodules: list of dict, her biri:
            - center: (y, x) merkez koordinati
            - area: piksel alani
            - bbox: (y1, x1, y2, x2) bounding box
            - diameter_px: yaklasik cap (piksel)
    """
    # Bos maske kontrolu
    if pred_mask is None or pred_mask.max() == 0:
        return []

    labeled, n_components = ndimage.label(pred_mask.astype(np.int32))
    nodules = []

    for i in range(1, n_components + 1):
        component = (labeled == i)
        area = int(component.sum())

        if area < min_area:
            continue

        # Merkez
        center = ndimage.center_of_mass(component)

        # Bounding box
        rows = np.any(component, axis=1)
        cols = np.any(component, axis=0)
        row_indices = np.where(rows)[0]
        col_indices = np.where(cols)[0]

        if len(row_indices) == 0 or len(col_indices) == 0:
            continue

        y1, y2 = int(row_indices[0]), int(row_indices[-1])
        x1, x2 = int(col_indices[0]), int(col_indices[-1])

        # Yaklasik cap (daire varsayimi)
        diameter = np.sqrt(area / np.pi) * 2

        nodules.append({
            'center': (float(center[0]), float(center[1])),
            'area': area,
            'bbox': (y1, x1, y2, x2),
            'diameter_px': float(diameter),
        })

    return nodules


# Validasyon seti uzerinde connected component analizi
n_test = min(100, len(val_dataset))
total_nodules_found = 0
total_gt_nodules = 0
all_pred_areas = []
all_gt_areas = []

with torch.no_grad():
    for i in range(n_test):
        img, gt_mask = val_dataset[i]
        img_gpu = img.unsqueeze(0).to(device)

        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = model(img_gpu)
        pred_mask = (torch.sigmoid(pred) > 0.5).cpu().numpy()[0, 0]
        gt_np = gt_mask.numpy()[0]

        pred_nodules = extract_nodules_from_mask(pred_mask)
        gt_nodules = extract_nodules_from_mask(gt_np)

        total_nodules_found += len(pred_nodules)
        total_gt_nodules += len(gt_nodules)

        all_pred_areas.extend([n['area'] for n in pred_nodules])
        all_gt_areas.extend([n['area'] for n in gt_nodules])

print(f"Connected Component Analizi (ilk {n_test} dilim):")
print(f"  GT nodul sayisi:         {total_gt_nodules}")
print(f"  Tahmin edilen nodul:     {total_nodules_found}")
if total_gt_nodules > 0:
    detection_rate = min(total_nodules_found, total_gt_nodules) / total_gt_nodules
    print(f"  Tespit orani (approx):   {detection_rate * 100:.1f}%")
if all_pred_areas:
    print(f"  Tahmin nodul alan ort:   {np.mean(all_pred_areas):.1f} px")
if all_gt_areas:
    print(f"  GT nodul alan ort:       {np.mean(all_gt_areas):.1f} px")

---
## 10. Final Değerlendirme ve Kaydetme

In [ ]:
# Tam validasyon seti uzerinde final metrikler
model.eval()
all_dices = []
all_ious = []
all_precisions = []
all_recalls = []

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(images)

        metrics = compute_metrics(outputs.float(), masks.float())
        all_dices.append(metrics['dice'])
        all_ious.append(metrics['iou'])
        all_precisions.append(metrics['precision'])
        all_recalls.append(metrics['recall'])

final_dice = np.mean(all_dices)
final_iou = np.mean(all_ious)
final_precision = np.mean(all_precisions)
final_recall = np.mean(all_recalls)

print("=" * 60)
print("FINAL VALIDASYON SONUCLARI")
print("=" * 60)
print(f"  Dice Score:  {final_dice:.4f} +/- {np.std(all_dices):.4f}")
print(f"  IoU:         {final_iou:.4f} +/- {np.std(all_ious):.4f}")
print(f"  Precision:   {final_precision:.4f} +/- {np.std(all_precisions):.4f}")
print(f"  Recall:      {final_recall:.4f} +/- {np.std(all_recalls):.4f}")

# Metrikleri CSV kaydet
metrics_df = pd.DataFrame({
    'metric': ['dice', 'iou', 'precision', 'recall'],
    'mean': [final_dice, final_iou, final_precision, final_recall],
    'std': [
        np.std(all_dices), np.std(all_ious),
        np.std(all_precisions), np.std(all_recalls)
    ],
})
metrics_df.to_csv(OUTPUT_DIR / 'ct_seg_metrics.csv', index=False)
print(f"\nMetrikler kaydedildi: {OUTPUT_DIR / 'ct_seg_metrics.csv'}")

# History CSV
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / 'ct_seg_training_history.csv', index=False)
print(f"Egitim historisi:    {OUTPUT_DIR / 'ct_seg_training_history.csv'}")

# Model boyutu
model_path = OUTPUT_DIR / 'ct_seg_best_model.pth'
if model_path.exists():
    model_size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f"\nModel dosyasi:       {model_path} ({model_size_mb:.1f} MB)")

In [ ]:
# --- Ozet Rapor ---
print("\n" + "=" * 65)
print("AlpCAN CT Pipeline -- Nodul Segmentasyon Egitimi OZET")
print("=" * 65)

print(f"\n--- Dataset ---")
print(f"  LIDC-IDRI (zhangweiled/lidcidri)")
print(f"  Toplam dilim:  {len(df):,}")
print(f"  Train: {len(train_df):,} dilim ({len(train_patients)} hasta)")
print(f"  Val:   {len(val_df):,} dilim ({len(val_patients)} hasta)")
print(f"  Maske: Konsensus (>= %50 annotator uyumu)")

print(f"\n--- Model ---")
print(f"  Mimari:    U-Net (ResNet-34 encoder, ImageNet pre-trained)")
print(f"  Giris:     {IMG_SIZE}x{IMG_SIZE}x3")
print(f"  Parametre: {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"  Loss:      DiceBCE (0.5 + 0.5)")
print(f"  Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print(f"  Scheduler: CosineAnnealing (T_max={NUM_EPOCHS})")
print(f"  Epoch:     {NUM_EPOCHS}")
print(f"  AMP:       {'ON' if use_amp else 'OFF'}")

print(f"\n--- Performans ---")
print(f"  Best Epoch:  {best_epoch}")
print(f"  Val Dice:    {final_dice:.4f}")
print(f"  Val IoU:     {final_iou:.4f}")
print(f"  Val Prec:    {final_precision:.4f}")
print(f"  Val Recall:  {final_recall:.4f}")

print(f"\n--- Cikti Dosyalari ---")
for f in sorted(OUTPUT_DIR.glob('ct_seg_*')):
    size_kb = f.stat().st_size / 1024
    if size_kb >= 1024:
        print(f"  {f.name} ({size_kb / 1024:.1f} MB)")
    else:
        print(f"  {f.name} ({size_kb:.1f} KB)")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. Model agirliklarini AlpCAN backend'e entegre et")
print(f"  2. nodule_detection_inference.py guncelle")
print(f"  3. Nodul karakterizasyon modeli egit (Notebook 07)")
print(f"  4. 3D volumetrik segmentasyon (tam BT serisi)")
print("=" * 65)